# 05 — Embeddings, Vector Stores & Retrieval (RAG)
### The knowledge retrieval half of SupportPilot

This notebook covers LangChain's `Embeddings` interface, `FAISS` vector
stores, and `Retriever`s — the machinery behind `retrieval.py`'s knowledge
base search. This is the same RAG pattern from Week 3 of the course
(embeddings -> vector DB -> retriever), applied to the 5 ShopStream India
policy documents in `kb/`.

## 5.1 The `Embeddings` interface

Every embedding provider in LangChain (OpenAI, HuggingFace, Cohere, ...)
implements the same two methods:

- `embed_documents(texts: list[str]) -> list[list[float]]` — embed many texts
  at once (for indexing)
- `embed_query(text: str) -> list[float]` — embed a single text (for
  searching)

This project uses a **custom, offline** implementation
(`embeddings.py`'s `LocalTfidfEmbeddings`) backed by scikit-learn's TF-IDF
instead of a real embedding model, specifically so the whole pipeline runs
with no API key and no model download. Let's see it satisfy the interface.

In [ ]:
from embeddings import LocalTfidfEmbeddings

emb = LocalTfidfEmbeddings()

corpus = [
    "Refunds are processed within 5-7 business days.",
    "Damaged items can be replaced free of charge within 48 hours.",
    "Payment failures usually reverse automatically within 48 hours.",
]

emb.fit(corpus)   # TF-IDF needs to see the vocabulary before it can embed anything
vectors = emb.embed_documents(corpus)

print(f"Number of vectors: {len(vectors)}")
print(f"Vector dimensionality: {len(vectors[0])}")   # one dimension per vocabulary word


In [ ]:
query_vector = emb.embed_query("how long does a refund take")
print(f"Query vector dimensionality: {len(query_vector)}")   # same dimensionality -- required for comparison


## 5.2 Why dimensionality has to match — and why `fit` matters

A vector store compares a query vector to document vectors using something
like cosine similarity, which only makes sense if both vectors live in the
same space. That's why `LocalTfidfEmbeddings.fit()` has to run on the
**combined** vocabulary before embedding anything — if you fit separately on
different corpora, the vector dimensions wouldn't line up (or would even
represent different words at the same index).

This is exactly why `retrieval.py`'s `KnowledgeRetriever.__init__` fits the
embeddings object on **all** documents (public + internal) even though it
builds two separate FAISS stores:

```python
public_embeddings = LocalTfidfEmbeddings()
public_embeddings.fit([d.page_content for d in all_docs])  # shared vocabulary
```

A real embedding model (OpenAI, HuggingFace) doesn't have this constraint —
its vocabulary is fixed by pretraining, not by whatever corpus you hand it —
which is one of the practical advantages of swapping in a real model once
you have API access.

## 5.3 Documents and chunking

Before embedding, text has to be split into retrievable units — LangChain's
`Document` object pairs `page_content` with `metadata` you can use for
citation. `retrieval.py`'s `_load_section_documents()` splits each policy
markdown file on `## ` headers, attaching `source_doc` and `section`
metadata to each chunk.

In [ ]:
from retrieval import _load_section_documents

docs = _load_section_documents()
print(f"Loaded {len(docs)} section-level chunks\n")

for d in docs[:3]:
    print(f"[{d.metadata['source_doc']} — {d.metadata['section']}]")
    print(d.page_content[:120].replace(chr(10), ' '), "...")
    print()


## 5.4 Building a FAISS vector store

`FAISS.from_documents(docs, embeddings)` embeds every document and builds a
searchable index in one call. `.as_retriever()` then wraps that index behind
LangChain's standard `Retriever` interface — the same interface a
`ChatPromptTemplate` or a RAG chain expects, regardless of which vector
store backs it.

In [ ]:
from langchain_community.vectorstores import FAISS

texts = [d.page_content for d in docs]
index_embeddings = LocalTfidfEmbeddings()
index_embeddings.fit(texts)

store = FAISS.from_documents(docs, index_embeddings)
retriever = store.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke("customer wants a refund for a damaged item")
for r in results:
    print(f"{r.metadata['source_doc']} — {r.metadata['section']}")
    print(r.page_content[:150].replace(chr(10), ' '), "...")
    print()


## 5.5 The real thing: `retrieval.py`'s `KnowledgeRetriever`

The project wraps two of these (public + internal-only) behind one class, so
the customer-facing retriever can never surface the internal escalation
criteria document, while the escalation/validation agents can still query it
separately.

In [ ]:
from retrieval import KnowledgeRetriever

kr = KnowledgeRetriever()

print("Customer-facing search:")
for r in kr.retrieve("my package arrived broken"):
    print(f"  {r['source_doc']} — {r['section']}")

print("\nInternal-only search (never shown to a customer):")
for r in kr.retrieve_internal("unrecognized transaction on account"):
    print(f"  {r['source_doc']} — {r['section']}")


Notice the internal search surfaces the "Always Escalate" section of
`escalation_criteria.md` — that's what grounds the validation/escalation
agents' reasoning about *why* something needs a human, without that
reasoning ever leaking into a customer-facing draft.

## 5.6 Swapping in real embeddings

Because everything above is coded against the `Embeddings` interface, not
against `LocalTfidfEmbeddings` specifically, switching to real semantic
embeddings is a one-line change:

```python
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

store = FAISS.from_documents(docs, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 3})
```

Everything downstream — `tools.py`'s `retrieve_kb_tool`, `chains.py`'s
drafting chain, `pipeline.py`'s orchestration — needs zero changes, because
none of it depends on *how* the embeddings were computed, only on the
`Retriever` interface they're wrapped behind. This is the practical payoff
of coding against interfaces instead of concrete classes, and it's worth
sitting with as a design lesson on its own.

## Exercise

1. Add a 6th policy document to `kb/` (e.g. `loyalty_program.md` with a
   couple of `## ` sections) describing a fictional ShopStream loyalty
   points policy.
2. Re-run `KnowledgeRetriever()` and query it with something a loyalty-point
   question would use — confirm your new sections show up in results.
3. TF-IDF is a *keyword* similarity measure, not a semantic one. Write a
   query using different words than your document but the same meaning
   (e.g. document says "points expire after 12 months", query asks "when do
   my rewards go bad") and see whether TF-IDF still finds it. What does this
   tell you about when TF-IDF retrieval is good enough vs. when you'd need
   real embeddings?